In [ ]:
import numpy as np
from stl import mesh as stl_mesh
import plotly.graph_objects as go  # for visualization
import os
import trimesh


import triply
from triply.utils import reload_all
from triply import logger, set_log_level
set_log_level("DEBUG")
#reload_all()

working_path = os.getcwd()
print("Current working directory:", working_path)

In [ ]:
# size of domain
pz2 = 20000
py2 = 20000
px2 = 20000

# --- Discretization of the domain ---
# Resolution in each axis, calculated as a funcion of the size of the gyroid's unit cell
dx_grid = px2 / 500
dy_grid = py2 / 500
dz_grid = pz2 / 500

# 1D coordinate arrays. np.arange(stop + step, step) includes the endpoint like MATLAB's colon with step.
x1 = np.arange(0, px2 + dx_grid, dx_grid)       # x positions from 0 to pz2 + dx_grid, and the step size is dx_grid
y1 = np.arange(0, py2 + dy_grid, dy_grid)       # y positions from 0 to py2, step dy_grid
z1 = np.arange(0, pz2 + dz_grid, dz_grid)       # z positions from 0 to pz2, step dz_grid

# Create 3D coordinate grids. indexing='ij' -> (X,Y,Z) follow x1,y1,z1 order like MATLAB.
x, y, z = np.meshgrid(x1, y1, z1, indexing='ij')
print(np.min(x),np.max(x))
print(np.min(y),np.max(y))
print(np.min(z),np.max(z))

print(f"x-axis resolution {np.size(x1)=}, y-axis resolution {np.size(y1)=}, z-axis resolution {np.size(z1)=}")
print(f"In total, {np.size(x)} voxels in the 3D grid")

In [ ]:
# --------- Create gyroids -----------
file_name = "gyroid-test"

# ---  Define Period of gyroid unit cell -------
# px: linear gradient along x + sinusoidal ripple driven by y
px = 3000
# py: linear gradient along y + sinusoidal ripple driven by z
py = 3000 + 9000 * z / pz2
# pz: radial bull's-eye pattern — largest period at the centre of the xy-plane
pz = 3000 + 9000 * z / pz2  # scale to 3000-12000 range


# ------- check for errors -------
#in the period of the gyroid is too small for the grid resolution, it will cause errors in the marching cubes algorithm
"""if np.min(pz) <= 5 * dz_grid :
    print(f"max pz: {np.max(pz)}, min pz: {np.min(pz)}, resultion dz_grid: {dz_grid}")
    raise ValueError("Period too small for the grid resolution. z-axis.")   
elif np.min(py) <= 5 * dy_grid :
    print(f"max py: {np.max(py)}, min py: {np.min(py)}, resultion dy_grid: {dy_grid}")
    raise ValueError("Period too small for the grid resolution. y-axis.")   
elif np.min(px) <= 5 * dx_grid :
    print(f"max px: {np.max(px)}, min px: {np.min(px)}, resultion dx_grid: {dx_grid}")
    raise ValueError("Period too small for the grid resolution. x-axis.")"""

# ------ Define Thickness function t ---   #can be from -1.4265847744427516 + 1.4265847744427516
t = np.zeros_like(x) + 350   # must be >> voxel spacing (dx_grid=50) or compute_field(mode="distance") keeps nothing

print('1')
# --- Gyroid scalar field v (isosurface at v=0 gives the gyroid surface) ---
term = ( np.sin((2*np.pi/px)*x) * np.cos((2*np.pi/py)*y)
    + np.sin((2*np.pi/py)*y) * np.cos((2*np.pi/pz)*z)
    + np.sin((2*np.pi/pz)*z) * np.cos((2*np.pi/px)*x) )  
print('2')

my_TPMS = triply.tpms_custom.CustomTPMSModel(x, y, z, t, term)
my_TPMS.compute_field(mode = "distance");

# --- Visualize result ---
triply.viz.twod_view_of_matrix(my_TPMS.v, x1, y1, z1, 0, 0.01)


In [ ]:
vertex, faces = triply.mesh_tools.mesh_from_matrix(matrix = my_TPMS.v,
                                         iso_level = 0.1,
                                         algo_step_size=2,
                                         x=my_TPMS.x,
                                         y=my_TPMS.y,
                                         z=my_TPMS.z,)

vertex, faces = triply.mesh_tools.keep_largest_connected_component(vertex, faces)
vertex, faces = triply.mesh_tools.auto_smooth_mesh(verts=vertex, faces=faces, smoothing_factor=0.8, improvement_tol=0.005)
vertex, faces = triply.mesh_tools.simplify_mesh(vertex, faces, target=3000000)
#vertex, faces = triply.mesh_tools.auto_smooth_mesh(verts=vertex, faces=faces, smoothing_factor=0.85, improvement_tol=0.005)
triply.viz.save_mesh_as_html(faces = faces,
                      verts = vertex,
                      file_name="C:\\Users\\cofo\\Desktop\\gyroids generation\\original_nop",
                      save = True)


In [ ]:
set_log_level("INFO")
reload_all()

solid=  np.copy(my_TPMS.v)
solid[solid > 0] = 1
solid[solid <= 0] = 0

test_matrice = triply.voxel_tools.detect_overhangs(solid, x1, y1, z1, angle = 45, bridge=30, add_support_voxels=True)

#invert x and z axis to match the orientation of the gyroid in the 3D view
rotated_matrix = np.swapaxes(test_matrice, 1, 2)

triply.viz.twod_view_of_matrix(rotated_matrix, x1, y1, z1, 0, 4)



In [ ]:
reload_all()
from triply import logger, set_log_level
set_log_level("INFO")

best_print_matrix, new_x, new_y, new_z = triply.voxel_tools.find_optimal_orientation(solid, x1, y1, z1, n=2, overhang_angle = 65, bridge_size=30, grid_sample_factor=0.5)

rotated_matrix = np.swapaxes(best_print_matrix, 1, 2)

triply.viz.twod_view_of_matrix(rotated_matrix, new_x, new_z, new_y, 0, 4)

In [ ]:
reload_all()
set_log_level("DEBUG")

to_smooth = np.copy(rotated_matrix)
#to_smooth[to_smooth > 1] = 0

vertex, faces = triply.mesh_tools.mesh_from_matrix(matrix = to_smooth,
                                         iso_level = 0.1,
                                         algo_step_size=1,
                                         x=new_x,
                                         y=new_z,
                                         z=new_y,)

vertex, faces = triply.mesh_tools.keep_largest_connected_component(vertex, faces)
vertex, faces = triply.mesh_tools.auto_smooth_mesh(verts=vertex, faces=faces, smoothing_factor=0.9, improvement_tol=0.005)
vertex, faces = triply.mesh_tools.simplify_mesh(vertex, faces, target=1000000)


triply.viz.save_mesh_as_html(faces = faces,
                      verts = vertex,
                      file_name="C:\\Users\\cofo\\Desktop\\gyroids generation\\ntest",
                      save = True)

In [ ]:
vertex,faces = triply.mesh_tools.fix_mesh(verts=vertex, faces=faces)

In [ ]:
triply.mesh_tools.export_as_STL(verts=vertex, faces=faces, path="C:\\Users\\cofo\\Desktop\\gyroids generation\\nop.stl")

